# 동일 개체 그룹에 ID를 부여하고 그래프에 적재하는 과제 LV2

function_overview, scatterplot, kdeplot, 릴리스 안내의 실제 문서 4편에서 추출한 트리플 14행을 사용합니다.  
**원문 판정 → 그룹 구성 → 그룹별 ID 부여 → 트리플에 ID 적용 → ER 품질 측정 → DB 적재와 통합**을 이어서 수행합니다.  
동일 개체 판정이 끝난 그룹에는 파이썬으로 ID를 부여합니다. LLM으로 ID를 다시 고르지 않습니다.  
충돌과 보류 처리는 실제 자료를 바꾸지 않고 별도의 검사 입력으로 확인합니다.  
1~8번은 파일과 파이썬으로 수행하며 API 키가 필요하지 않습니다.  
9번 직전에 실습용 Neo4j에 연결하고, 10번에서 APOC로 이미 저장된 중복 노드를 통합합니다.  


In [ ]:
# [제공코드] JSONL 자료를 읽고 이름과 출현 기록을 비교할 도구를 준비합니다.
import json
from pathlib import Path
from itertools import combinations
from difflib import SequenceMatcher
from pprint import pprint

data_dir = Path("data")

def load_rows(filename):
    """한 줄에 한 기록이 저장된 JSONL 파일을 딕셔너리 목록으로 읽습니다."""
    lines = (data_dir / filename).read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line.strip()]

def pair_key(left_id, right_id):
    """비교 순서가 바뀌어도 같은 두 기록을 같은 키로 나타냅니다."""
    return tuple(sorted((left_id, right_id)))


## 저장된 추출 결과를 살펴봅니다

이 자료는 실제 Seaborn 문서에서 LLM이 추출해 저장한 트리플의 일부입니다.  
원문에서 트리플을 다시 추출하지 않고, 저장된 주어와 목적어를 정리합니다.  
**출현 기록**은 트리플 한 행의 주어 또는 목적어 자리를 따로 기록한 것입니다.  
`triple_id`는 추출 행의 ID, `mention_id`는 그 행의 어느 자리인지 구분하는 출현 ID입니다.  

| 출처 키 | 뜻 |
|---|---|
| source_file | 원본 추출 JSONL 파일 경로 |
| source_line | 그 파일의 행 번호. 1부터 시작 |
| source_triple_index | 해당 행 안의 트리플 번호. 1부터 시작 |

`evidence`와 `source_doc_id`는 원문으로 돌아가 판정 근거를 확인할 때 사용합니다.  


In [ ]:
# [제공코드] kg_lv2_triples.jsonl: 원문과 추출 위치가 보존된 과제용 트리플입니다.
task_triples = load_rows("kg_lv2_triples.jsonl")
triple_by_id = {row["triple_id"]: row for row in task_triples}
original_triples = [dict(row) for row in task_triples]

# kg_corpus.jsonl: 추출의 출처인 실제 문서의 본문과 URL입니다.
task_documents = {row["doc_id"]: row for row in load_rows("kg_corpus.jsonl")}
first_document = task_documents[task_triples[0]["source_doc_id"]]
print("첫 원문:", first_document["title"], first_document["url"])
print(first_document["text"][:500])

print("추출 트리플:", len(task_triples))
for row in task_triples:
    print(row["triple_id"], row["subject"], row["relation"], row["object"])
pprint(task_triples[0])


In [ ]:
# [제공코드] 같은 이름이라도 트리플의 양 끝에서 등장한 기록은 각각 보존합니다.
mentions = []
for triple in task_triples:
    for role in ["subject", "object"]:
        mentions.append({
            "mention_id": triple["triple_id"] + ":" + role,
            "triple_id": triple["triple_id"], "role": role,
            "name": triple[role], "entity_type": triple[role + "_type"],
            "source_doc_id": triple["source_doc_id"], "evidence": triple["evidence"],
        })
print("출현 기록:", len(mentions))
pprint(mentions[:2])


## 1. 연결 입력의 출현 ID와 원본 필드를 검사합니다

**배경**: 다른 추출 버전의 출현 기록을 연결하면 이름이 같아도 잘못된 관계가 만들어집니다.  

**요구사항**  
- **validate_mentions(rows, triples)** 함수를 작성하세요. rows와 triples는 딕셔너리 목록입니다. triples의 각 행에서 주어·목적어의 출현 ID를 만들고, rows가 그 전체를 정확히 한 번씩 포함하는지 검사하세요. rows 순서는 달라도 허용합니다. 필드가 하나라도 빠졌거나 남았으면 값이 맞아도 거부합니다.
- **validate_mentions** 는 각 행의 `mention_id`, `triple_id`, `role`, `name`, `entity_type`, `source_doc_id`, `evidence` 7개 필드가 인자 triples에서 재구성한 출현 기록과 정확히 일치하는지 검사합니다. 누락·중복·범위 밖 ID·값 변경·추가 필드는 ValueError를 발생시키고, 정상이면 True를 반환하세요.
- **input_valid** 에 mentions와 task_triples를 검사한 결과를 담으세요. 에러 메시지는 자유입니다.

**확인 기준**: 정상 입력은 True입니다. 같은 ID의 행을 두 번 넣거나 원문을 변경하면 ValueError입니다.  

<details><summary>힌트</summary>

```text
접근방법:
- ID별 원본 기록을 만들고 전체 범위와 각 행을 대조합니다.

세부구현:
1. 트리플의 두 끝에서 기대하는 출현 기록을 만듭니다.
2. 출현 ID의 중복과 누락을 검사합니다.
3. 해당 ID의 원래 7개 필드를 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert input_valid is True, "정상 입력은 True를 반환하세요."
assert validate_mentions(list(reversed(mentions)), task_triples) is True, "입력 순서 변경은 허용하세요."
bad_inputs = [mentions[:-1], mentions + [dict(mentions[0])]]
for field in mentions[0]:
    changed = [dict(row) for row in mentions]
    changed[0][field] = "검사용 변경 값"
    bad_inputs.append(changed)
extra = [dict(row) for row in mentions]
extra[0]["추가키"] = True
bad_inputs.append(extra)
for rows in bad_inputs:
    rejected = False
    try:
        validate_mentions(rows, task_triples)
    except ValueError:
        rejected = True
    assert rejected, "누락·중복·범위 밖 ID·원본 변경·추가 필드를 거부하세요."
print("✅ 통과!")


### 그룹으로 묶을 판정 기록을 읽습니다

이 파일은 **기존 출현별 원문 검토를 같은 개체 쌍으로 바꾼 자료**입니다.  
검색 후보와 달리 원문 판정이 끝난 쌍이며, 양쪽 판정 이유도 남아 있습니다.  
새 모델 호출이나 골드 조회 없이 이 판정에서 그룹을 만들고 ID를 부여합니다.  


In [ ]:
# [제공코드] kg_lv2_pair_review.json: 이 과제 출현의 원문 검토와 같은 개체 쌍입니다.
pair_review = json.loads((data_dir / "kg_lv2_pair_review.json").read_text(encoding="utf-8"))
pair_decisions = pair_review["decisions"]

# 다른 추출 버전의 판정을 잘못 적용하지 않도록 전체 출현을 대조합니다.
assert pair_review["mentions"] == mentions, "판정 파일의 출현과 과제 입력이 다릅니다."
print("판정 출처:", pair_review["source"])
print("확인된 같은 개체 쌍:", len(pair_decisions))
pprint(pair_decisions[:2])


In [ ]:
# [제공코드] 같다고 확인한 쌍을 그룹으로 모읍니다. 연결되지 않은 기록도 한 개짜리 그룹으로 남깁니다.
def group_pairs(mention_ids, same_pairs):
    """같은 개체로 판정한 쌍을 이어 출현 그룹을 만듭니다.

    Args:
        mention_ids (list[str]): 보존할 전체 출현 ID.
        same_pairs (list[tuple[str, str]]): 같은 개체로 판정한 출현 ID 쌍.

    Returns:
        list[list[str]]: 정렬한 그룹 목록. 연결되지 않은 출현도 단독 그룹으로 남습니다.
    """
    groups = [{mention_id} for mention_id in mention_ids]
    for left_id, right_id in same_pairs:
        # 평가 범위 밖의 ID를 잘못 넣으면 기록이 빠질 수 있으므로 먼저 확인합니다.
        if left_id not in mention_ids or right_id not in mention_ids:
            raise ValueError("동일 판정 쌍에 입력 목록에 없는 ID가 있습니다.")
        joined = set()
        remaining = []
        for group in groups:
            if left_id in group or right_id in group:
                joined.update(group)
            else:
                remaining.append(group)
        remaining.append(joined)
        groups = remaining
    return sorted([sorted(group) for group in groups])

# 입력 예시: 두 초판 표기는 같은 책이고 개정판은 별도 책입니다.
example_ids = ["d01:object", "d02:object", "d03:object"]
example_pairs = [("d01:object", "d02:object")]  # 파이썬 입문과 파이썬 입문서

# 호출: 같은 초판끼리 묶고 개정판(d03:object)은 따로 남깁니다.
print(group_pairs(example_ids, example_pairs))


## 2. 판정된 쌍에 대칭성과 이행성을 적용합니다

**배경**: 원문에서 같은 대상으로 확인한 쌍을 따라 전체 출현을 동일 개체 그룹으로 묶습니다.  

**요구사항**  
- **direct_pairs** 에 pair_decisions 중 decision이 같음인 두 ID를 pair_key로 정렬한 튜플로 담으세요. 집합을 사용합니다.
- **mention_ids** 에 mentions의 mention_id를 입력 순서대로 담으세요.
- **er_groups** 에 group_pairs로 mention_ids를 direct_pairs에 따라 묶은 결과를 담으세요. 단독 출현도 포함합니다.
- **expanded_pairs** 에 er_groups 각 그룹 내부에서 가능한 모든 쌍을 pair_key로 정렬한 튜플의 집합으로 담으세요.
- **symmetry_ok** 에 direct_pairs의 모든 쌍에서 두 ID 순서를 바꿔 pair_key로 처리해도 같은 튜플인지 나타내는 bool을 담으세요.

**확인 기준**: 전체 출현 28건을 14개 그룹에 한 번씩 배정합니다. 직접 판정한 14쌍이 이행성에 따라 그룹 내부 37쌍으로 확장됩니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 그룹을 만드는 연결 쌍과 그룹 내부에서 평가할 전체 쌍을 구분합니다.

세부구현:
1. 같음 판정의 ID 쌍만 모읍니다.
2. 전체 출현 ID로 그룹을 만듭니다.
3. 각 그룹에서 combinations로 모든 쌍을 만듭니다.
4. 쌍의 방향을 바꿔도 같은지 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_direct = set()
for row in pair_decisions:
    if row["decision"] == "같음":
        expected_direct.add(pair_key(row["left_id"], row["right_id"]))
assert direct_pairs == expected_direct, "원문에서 같음으로 판정한 쌍만 모으세요."
assert mention_ids == [row["mention_id"] for row in mentions], "단독 출현도 전체 범위에 포함하세요."
assert er_groups == group_pairs(mention_ids, expected_direct), "판정 쌍을 따라 그룹을 만드세요."
expected_expanded = set()
for group in er_groups:
    for left_id, right_id in combinations(group, 2):
        expected_expanded.add(pair_key(left_id, right_id))
assert expanded_pairs == expected_expanded, "그룹 내부 모든 쌍을 포함하세요."
assert len(er_groups) == 14 and len(expanded_pairs) == 37, "전체 출현의 그룹과 내부 쌍 수를 확인하세요."
assert isinstance(symmetry_ok, bool) and symmetry_ok, "같은 개체 쌍은 방향을 구분하지 않습니다."
print("✅ 통과!")


### 판정과 충돌하는 그룹을 검사합니다

`hold_pair_conflicts`는 한 그룹 안에 다름 또는 보류 판정이 있으면 그 그룹을 보류합니다.  
아래 `proposed_groups`는 이 검사를 연습하려고 **서로 다른 함수 그룹을 일부러 합친 가상 오류 입력**입니다.  
실제 원문 판정과 `er_groups`는 변경하지 않습니다.  


In [ ]:
# [제공코드] 교안 01의 판정은 원본과 비교한 뒤 사용합니다. 검색 후보 자체는 그룹으로 묶지 않습니다.
def hold_pair_conflicts(groups, decisions):
    """같은 그룹 안에 다름·보류 판정이 있으면 그룹 전체를 보류합니다.

    Args:
        groups (list[list[str]]): 같음 판정으로 만든 출현 그룹.
        decisions (list[dict]): 출현 쌍의 같음·다름·보류 판정.

    Returns:
        tuple: (충돌 없는 그룹, 보류 그룹).
    """
    # 각 출현이 속한 그룹 번호를 기록해 두면 두 출현의 소속을 바로 비교할 수 있습니다.
    group_of = {}
    for group_index, group in enumerate(groups):
        for mention_id in group:
            group_of[mention_id] = group_index

    held_indices = {group_of[row["left_id"]] for row in decisions
                    if row["decision"] != "같음"
                    and group_of[row["left_id"]] == group_of[row["right_id"]]}
    return ([group for index, group in enumerate(groups) if index not in held_indices],
            [group for index, group in enumerate(groups) if index in held_indices])

# 입력 예시: 초판 두 표기를 같다고 묶었지만 별도 판정은 보류입니다.
pair_conflict_example_groups = [["d01:object", "d02:object"], ["d03:object"]]
pair_conflict_example_decisions = [{"left_id": "d01:object", "right_id": "d02:object", "decision": "보류"}]
print(hold_pair_conflicts(pair_conflict_example_groups, pair_conflict_example_decisions)[1])


In [ ]:
# [제공코드] histplot과 kdeplot은 서로 다른 함수입니다. 두 그룹을 잘못 합친 경우만 별도로 만듭니다.
conflict_pair = pair_key("t09:object", "t10:object")
mixed_group = []
proposed_groups = []
for group in er_groups:
    if set(group) & set(conflict_pair):
        mixed_group.extend(group)
    else:
        proposed_groups.append(list(group))
mixed_group.sort()
proposed_groups.insert(0, mixed_group)
conflict_decisions = list(pair_decisions) + [{
    "left_id": conflict_pair[0], "right_id": conflict_pair[1],
    "decision": "다름", "reason": "원문에서 histplot과 kdeplot을 각각 호출합니다.",
}]
print("잘못 합친 검사 그룹:", mixed_group)


## 3. 그룹 내부의 충돌이 있으면 ID 부여를 보류합니다

**배경**: 이행성으로 연결했더라도 같은 그룹 안의 다름 판정을 무시하고 통합해서는 안 됩니다.  

**요구사항**  
- **accepted_groups, held_groups** 에 hold_pair_conflicts로 er_groups와 pair_decisions를 검사한 두 목록을 담으세요.
- **accepted_probe, held_probe** 에는 같은 함수로 proposed_groups와 conflict_decisions를 검사한 두 목록을 담으세요.
- **held_probe** 를 출력하고 어떤 다름 판정 때문에 보류했는지 conflict_pair와 함께 확인하세요. 이후 실제 적재에는 accepted_groups만 사용합니다.

**확인 기준**: 실제 자료에는 그룹 내부 충돌이 없어 held_groups가 비어 있습니다. 가상 오류 입력에서는 mixed_group 하나만 held_probe에 남고, 나머지 그룹은 accepted_probe에 그대로 보존됩니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 각 그룹 안에 두 끝이 모두 포함된 다름 또는 보류 판정이 있는지 검사합니다.

세부구현:
1. 실제 그룹과 원문 판정을 검사합니다.
2. 잘못 합친 복사본도 같은 함수로 검사합니다.
3. 두 결과를 섞지 않고 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert accepted_groups == er_groups and held_groups == [], "실제 자료의 충돌 없는 그룹을 보존하세요."
assert held_probe == [mixed_group], "다름 판정의 두 출현이 같은 그룹이면 해당 그룹을 보류하세요."
assert accepted_probe == proposed_groups[1:], "충돌하지 않은 나머지 그룹은 보존하세요."
assert len(er_groups) == 14, "가상 오류 입력으로 원래 그룹을 바꾸지 마세요."
print("✅ 통과!")


### 그룹에 ID 하나를 부여합니다

`assign_group_ids`는 그룹의 사전순 첫 출현 ID에 `entity:`를 붙여 ID를 만듭니다.  
같은 그룹의 모든 출현이 그 ID를 공유합니다. 함수에 전달하지 않은 보류 그룹의 출현은 `review`, `None`으로 남습니다.  


In [ ]:
# [제공코드] 같은 개체로 확정한 그룹에 ID 하나를 부여하고 원래 출현 기록에 붙입니다.
def assign_group_ids(mentions, groups):
    """같은 그룹에 같은 표준 ID를 붙인 출현 목록을 반환합니다.

    Args:
        mentions (list[dict]): 원래 출현 기록 전체.
        groups (list[list[str]]): 판정 충돌이 없는 그룹의 출현 ID 목록.

    Returns:
        list[dict]: 원본에 standard_id와 status를 추가한 복사본.
            그룹에 없는 출현은 ID 없이 review 상태로 남깁니다.
    """
    by_id = {row["mention_id"]: row for row in mentions}
    if len(by_id) != len(mentions):
        raise ValueError("출현 ID가 중복되었습니다.")

    id_by_mention = {}
    for group in groups:
        if not group or len(group) != len(set(group)):
            raise ValueError("그룹이 비었거나 같은 출현이 중복되었습니다.")
        if not set(group).issubset(by_id):
            raise ValueError("그룹에 원본에 없는 출현 ID가 있습니다.")
        types = {by_id[mention_id]["entity_type"] for mention_id in group}
        if len(types) != 1:
            raise ValueError("타입이 다른 출현의 판정을 다시 확인하세요.")

        # 정렬상 첫 출현 ID를 사용해 같은 그룹에 다시 실행해도 같은 ID를 만듭니다.
        standard_id = "entity:" + min(group)
        for mention_id in group:
            if mention_id in id_by_mention:
                raise ValueError("한 출현이 여러 그룹에 들어 있습니다.")
            id_by_mention[mention_id] = standard_id

    links = []
    for row in mentions:
        standard_id = id_by_mention.get(row["mention_id"])
        status = "linked" if standard_id is not None else "review"
        links.append(dict(row, standard_id=standard_id, status=status))
    return links


# 예시 입력: 두 표기는 같은 초판, 개정판은 별도 그룹입니다.
id_example_mentions = [
    {"mention_id": "d01:object", "name": "파이썬 입문", "entity_type": "Book"},
    {"mention_id": "d02:object", "name": "파이썬 입문서", "entity_type": "Book"},
    {"mention_id": "d03:object", "name": "파이썬 입문(개정판)", "entity_type": "Book"},
]
id_example_groups = [["d01:object", "d02:object"], ["d03:object"]]
for row in assign_group_ids(id_example_mentions, id_example_groups):
    print("이름:", row["name"], "/ 표준 ID:", row["standard_id"])


## 4. 확정된 그룹마다 하나의 표준 ID를 부여합니다

**배경**: 같은 개체인지 이미 판정했으므로 ID는 파이썬으로 한 번만 부여합니다.  

**요구사항**  
- **entity_links** 에 assign_group_ids로 mentions와 accepted_groups를 처리한 결과를 담으세요.
- **linked** 에 entity_links 중 status가 linked인 행의 `mention_id: standard_id`를 딕셔너리로 담으세요.
- **pending** 에 나머지 행의 mention_id를 입력 순서대로 담으세요.
- **probe_links** 에 같은 함수로 mentions와 accepted_probe를 처리한 결과를 담으세요. **probe_pending** 에는 probe_links 중 status가 review인 출현 ID를 입력 순서대로 담으세요. 이 검사 결과는 실제 entity_links에 섞지 않습니다.

**확인 기준**: 실제 자료의 28개 출현은 14개 ID에 연결되고 pending은 비어 있습니다. 가상 오류 입력의 probe_pending에는 mixed_group의 출현만 남아야 합니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 확정 그룹만 함수에 전달하되 원본 출현 목록은 전체를 전달합니다.

세부구현:
1. 실제 확정 그룹에 ID를 부여합니다.
2. 확정 ID 조회 사전과 보류 ID 목록을 만듭니다.
3. 검사용 확정 그룹에서도 보류 출현이 사라지지 않는지 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert entity_links == assign_group_ids(mentions, accepted_groups), "그룹 ID와 원본 필드를 그대로 보존하세요."
expected_linked = {}
for group in accepted_groups:
    for mention_id in group:
        expected_linked[mention_id] = "entity:" + min(group)
assert linked == expected_linked and pending == [], "같은 그룹은 같은 ID로 연결하고 상태를 구분하세요."
assert probe_links == assign_group_ids(mentions, accepted_probe), "검사용 보류 출현도 원문과 함께 남기세요."
expected_pending = [row["mention_id"] for row in mentions if row["mention_id"] in mixed_group]
assert probe_pending == expected_pending, "보류 그룹의 출현만 원래 입력 순서로 남기세요."
print("✅ 통과!")


### ID를 붙인 기록과 그룹을 검사합니다

`check_links`는 원본 출현의 보존과 ID 상태를 검사합니다.  
`groups_from_links`는 같은 ID의 출현을 모으며, ID가 없는 출현은 각각 단독 그룹으로 남깁니다.  


In [ ]:
# [제공코드] ID를 붙이는 과정에서 원본 출현을 빠뜨리거나 바꾸지 않았는지 검사합니다.
def check_links(mentions, links):
    """출현 보존, 연결 상태와 같은 ID의 타입 일관성을 검사합니다.

    Args:
        mentions (list[dict]): 원본 출현 전체.
        links (list[dict]): 표준 ID를 붙인 출현 전체.

    Returns:
        None: 검사를 통과하면 다음 코드로 진행합니다.
    """
    original = {row["mention_id"]: row for row in mentions}
    linked_ids = [row["mention_id"] for row in links]
    if len(original) != len(mentions) or len(linked_ids) != len(set(linked_ids)):
        raise ValueError("출현 ID가 중복되었습니다.")
    if set(linked_ids) != set(original):
        raise ValueError("원본 출현과 연결 기록의 범위가 다릅니다.")

    type_by_id = {}
    for row in links:
        for field, value in original[row["mention_id"]].items():
            if row.get(field) != value:
                raise ValueError("원본과 다른 필드: " + field)
        standard_id = row["standard_id"]
        if row["status"] == "review" and standard_id is None:
            continue
        if row["status"] != "linked" or not isinstance(standard_id, str) or not standard_id:
            raise ValueError("연결 상태와 표준 ID를 확인하세요.")
        previous_type = type_by_id.setdefault(standard_id, row["entity_type"])
        if previous_type != row["entity_type"]:
            raise ValueError("같은 표준 ID에 다른 타입이 섞였습니다.")


# 예시 입력: 파이썬 입문의 원래 기록에 그룹 ID만 추가합니다.
check_example_mentions = [{"mention_id": "d01:object", "name": "파이썬 입문", "entity_type": "Book"}]
check_example_links = [dict(check_example_mentions[0], status="linked", standard_id="entity:d01:object")]
check_links(check_example_mentions, check_example_links)
print("원본 출현과 ID 기록 검사 완료")


In [ ]:
# [제공코드] 같은 확정 ID의 기록을 묶고, 미확정 기록은 각각 따로 남깁니다.
def groups_from_links(links):
    """같은 확정 ID끼리 묶고 미확정 출현은 단독 그룹으로 남깁니다.

    Args:
        links (list[dict]): mention_id, status, standard_id가 있는 연결 목록.

    Returns:
        list[list[str]]: 그룹 내부와 그룹 목록을 정렬한 출현 ID 목록.
    """
    buckets = {}
    groups = []
    for row in links:
        if row["status"] != "linked":
            groups.append([row["mention_id"]])
            continue
        buckets.setdefault(row["standard_id"], []).append(row["mention_id"])
    groups.extend(buckets.values())
    # 그룹 내부와 그룹 목록을 모두 정렬해 입력 순서에 따른 차이를 없앱니다.
    return sorted(sorted(group) for group in groups)


# 예시 입력: d01과 d02의 책은 같은 초판, d03의 책은 개정판입니다.
grouping_example_links = [
    {"mention_id": "d01:object", "status": "linked", "standard_id": "entity:d01:object", "entity_type": "Book"},
    {"mention_id": "d02:object", "status": "linked", "standard_id": "entity:d01:object", "entity_type": "Book"},
    {"mention_id": "d03:object", "status": "linked", "standard_id": "entity:d03:object"},
]
# 예시 호출: 초판의 두 출현만 같은 그룹이 됩니다.
print(groups_from_links(grouping_example_links))


## 5. ID 부여 후에도 그룹과 원본이 같은지 확인합니다

**배경**: 새 ID를 붙이는 과정에서 그룹이 갈라지거나 서로 다른 그룹이 합쳐지지 않았는지 확인합니다.  

**요구사항**  
- **check_links** 에 mentions와 entity_links를 전달해 원본과 ID 상태를 검사하세요. 정상일 때는 반환값 없이 다음 줄로 진행합니다.
- **id_buckets** 에 linked의 표준 ID를 키로, 해당 출현 ID를 오름차순으로 정렬한 리스트를 값으로 담으세요.
- **identity_groups** 에 groups_from_links로 entity_links를 묶은 결과를 담으세요.
- **assignment_consistent** 에 identity_groups와 er_groups가 같은지 나타내는 bool을 담고 그룹 수와 함께 출력하세요.

**확인 기준**: 원래 14개 그룹이 그대로 유지되고 원래 이름, 타입, 출처와 근거도 보존됩니다. 그룹 ID의 문자열 모양은 실제 대상의 이름을 뜻하지 않습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 원본 필드 검증과 ID별 그룹 비교를 나누어 수행합니다.

세부구현:
1. 출현 기록과 ID 상태를 검사합니다.
2. ID별로 출현을 모아 정렬합니다.
3. ID로 다시 만든 그룹이 이전 판정 그룹과 같은지 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_buckets = {}
for mention_id, standard_id in linked.items():
    expected_buckets.setdefault(standard_id, []).append(mention_id)
for standard_id in expected_buckets:
    expected_buckets[standard_id].sort()
assert id_buckets == expected_buckets, "ID별 출현을 빠짐없이 정렬하세요."
assert identity_groups == groups_from_links(entity_links) == er_groups, "ID를 부여해도 그룹 소속은 그대로여야 합니다."
assert isinstance(assignment_consistent, bool) and assignment_consistent, "그룹 일치 여부를 bool로 담으세요."
print("✅ 통과!")


## 6. 양 끝에 표준 ID가 있는 트리플을 적재 대상으로 만듭니다

**배경**: 관계의 주어와 목적어가 어떤 표준 노드인지 정하고, 한쪽이라도 보류되면 원래 트리플을 따로 남깁니다.  

**요구사항**  
- **attach_ids(triples, mapping)** 함수를 작성하세요. triples는 트리플 딕셔너리 목록, mapping은 `출현 ID: 표준 ID` 사전입니다. 각 행의 triple_id에 `:subject`, `:object`를 붙여 양 끝을 찾습니다.
- **attach_ids** 는 양 끝이 mapping에 있으면 원본 복사본에 subject_id와 object_id를 추가하고, 한쪽이라도 없으면 원본 복사본을 보류 목록에 넣습니다. 두 목록 모두 입력 순서를 유지하며 `(적재 대상, 보류 목록)`을 반환합니다. 입력을 수정하지 않습니다.
- **normalized_triples, held_triples** 에 task_triples와 linked를 처리한 결과를 담으세요.
- **probe_mapping** 에 probe_links 중 status가 linked인 행의 `mention_id: standard_id`를 담고, **probe_triples, probe_held_triples** 에 task_triples와 probe_mapping을 처리한 결과를 담으세요.

**확인 기준**: 실제 자료의 14행에는 모두 두 ID가 붙고 held_triples는 비어 있습니다. 검사용 보류 그룹에 속한 출현이 있는 행은 probe_held_triples에 원문 그대로 남습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 트리플의 역할은 어느 ID를 찾을지 정할 뿐, 같은 개체인지 새로 판정하는 기준이 아닙니다.

세부구현:
1. 주어와 목적어의 출현 ID 두 개를 만듭니다.
2. 양 끝의 ID가 모두 있을 때만 두 ID를 복사본에 붙입니다.
3. 미해결 행은 별도 목록에 남깁니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
for mapping, actual, held in [(linked, normalized_triples, held_triples),
                              (probe_mapping, probe_triples, probe_held_triples)]:
    expected_connected, expected_held = [], []
    for triple in original_triples:
        subject_mention = triple["triple_id"] + ":subject"
        object_mention = triple["triple_id"] + ":object"
        if subject_mention in mapping and object_mention in mapping:
            expected = dict(triple, subject_id=mapping[subject_mention], object_id=mapping[object_mention])
            expected_connected.append(expected)
        else:
            expected_held.append(dict(triple))
    assert actual == expected_connected and held == expected_held, "양 끝의 ID와 원래 행의 모든 필드를 보존하세요."
    before = json.dumps([original_triples, mapping], ensure_ascii=False)
    assert attach_ids(original_triples, mapping) == (expected_connected, expected_held), "실제 자료와 보류 입력에 같은 함수를 적용하세요."
    assert json.dumps([original_triples, mapping], ensure_ascii=False) == before, "함수는 입력을 수정하지 않습니다."
assert probe_mapping == {row["mention_id"]: row["standard_id"] for row in probe_links if row["status"] == "linked"}, "보류 기록은 ID 조회 사전에 넣지 마세요."
assert len(normalized_triples) == 14 and held_triples == [], "실제 자료의 전체 14개 관계를 보존하세요."
assert probe_held_triples, "한쪽이라도 보류한 트리플은 삭제하지 않고 별도 목록에 남기세요."
assert task_triples == original_triples, "원본 트리플을 변경하지 마세요."
print("✅ 통과!")


## 7. 적재 노드와 관계에 원래 기록을 남깁니다

**배경**: 같은 개체를 노드 하나로 저장하면서 모든 별칭과 출현 근거를 보존합니다.  

**요구사항**  
- **used_mentions** 에 normalized_triples의 양 끝에 해당하는 mentions 기록을 원래 mentions 순서대로 복사해 담으세요.
- **node_rows** 에 표준 ID마다 `standard_id`, `canonical_name`, `entity_type`, `aliases`, `mention_ids`, `mentions` 여섯 키의 딕셔너리를 하나씩 담고 standard_id 순서로 정렬하세요.
- **node_rows** 의 canonical_name은 같은 ID 구성원 중 mention_id가 사전순으로 첫 번째인 출현의 name입니다. entity_type은 그 출현의 타입, aliases는 원래 name을 중복 없이 정렬한 목록, mention_ids는 정렬한 출현 ID 목록, mentions는 해당 출현을 원래 입력 순서대로 복사한 목록입니다.
- **relationship_rows** 에 normalized_triples의 각 행을 입력 순서대로 복사해 담으세요.

**확인 기준**: 노드 기록은 14개, 관계 기록은 14개입니다. 출현 ID 28개가 노드에 한 번씩 보존되고, 관계에는 두 ID와 원래 근거가 남습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 별도 개체 목록에서 ID를 고르지 않고 이미 부여한 ID로 원래 출현을 모읍니다.

세부구현:
1. 적재할 트리플의 출현만 고릅니다.
2. ID별로 출현 기록을 모읍니다.
3. 대표 이름과 전체 별칭 및 근거 목록을 만듭니다.
4. 관계 자료도 원본 필드를 보존해 복사합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
accepted_ids = {row["triple_id"] for row in normalized_triples}
expected_used = [row for row in mentions if row["triple_id"] in accepted_ids]
assert used_mentions == expected_used, "적재할 트리플의 출현을 입력 순서대로 보존하세요."
expected_node_ids = sorted({linked[row["mention_id"]] for row in expected_used})
assert [row["standard_id"] for row in node_rows] == expected_node_ids, "ID별 노드 하나를 정렬해 만드세요."
all_mention_ids = []
for node in node_rows:
    members = [row for row in expected_used if linked[row["mention_id"]] == node["standard_id"]]
    first = min(members, key=lambda row: row["mention_id"])
    expected = {"standard_id": node["standard_id"], "canonical_name": first["name"],
                "entity_type": first["entity_type"], "aliases": sorted({row["name"] for row in members}),
                "mention_ids": sorted(row["mention_id"] for row in members), "mentions": members}
    assert node == expected, "대표 이름과 타입, 모든 별칭과 원래 출현 기록을 보존하세요."
    all_mention_ids.extend(node["mention_ids"])
assert sorted(all_mention_ids) == sorted(row["mention_id"] for row in expected_used), "출현을 누락하거나 여러 노드에 배정하지 마세요."
assert relationship_rows == normalized_triples, "관계의 방향과 타입, 원래 모든 필드를 보존하세요."
assert len(node_rows) == len(relationship_rows) == 14, "전체 자료의 노드와 관계 수를 확인하세요."
print("✅ 통과!")


### 그룹 결과를 골드와 비교합니다

골드는 여기서 처음 읽으며 ID 부여나 그룹 구성에 사용하지 않습니다.  
직접 만든 ID는 골드의 ID 문자열과 달라도 정상입니다. **어떤 출현들이 함께 묶였는지**를 쌍으로 비교합니다.  


In [ ]:
# [제공코드] kg_gold.json: 전체 출현을 사람이 검토한 정답 그룹입니다. 평가에서만 사용합니다.
gold_data = json.loads((data_dir / "kg_gold.json").read_text(encoding="utf-8"))
evaluation_ids = {row["mention_id"] for row in mentions}
print("평가 범위:", len(evaluation_ids), "출현. 보류 검사에서도 같은 범위를 유지합니다.")


## 8. 정밀도와 재현율, F1과 오병합을 함께 평가합니다

**배경**: 올바르게 묶은 쌍, 잘못 묶은 쌍과 놓친 쌍을 같은 출현 범위에서 확인합니다.  

**요구사항**  
- **gold_pairs** 에 gold_data의 각 groups 항목에서 mention_ids를 evaluation_ids와 교집합한 뒤 가능한 모든 쌍을 담으세요. **predicted_pairs** 에는 identity_groups 내부의 모든 쌍을 담으세요. 각 쌍은 pair_key로 정렬한 튜플이며 두 산출물은 집합입니다.
- **evaluate_pairs(predicted, gold)** 함수를 작성해 tp, fp, fn, precision, recall, f1 여섯 키의 딕셔너리를 반환하세요. TP는 교집합, FP는 예측에만, FN은 골드에만 있는 쌍의 수입니다. 정밀도는 TP/(TP+FP), 재현율은 TP/(TP+FN), F1은 2TP/(2TP+FP+FN)입니다. 분모가 0이면 0.0을 씁니다. 입력을 수정하지 않습니다.
- **metrics** 에 predicted_pairs와 gold_pairs를 evaluate_pairs로 평가한 결과를 담으세요.
- **overmerge_pairs** 에 3번의 proposed_groups 내부의 모든 쌍을 담고, 같은 gold_pairs로 평가한 결과를 **overmerge_metrics** 에 담으세요.
- **held_pairs** 에 accepted_probe 내부의 모든 쌍을 담고, 같은 gold_pairs로 평가한 결과를 **held_metrics** 에 담으세요. 보류 출현을 evaluation_ids나 골드에서 제외하지 않습니다.
- **one_pair** 에 gold_pairs의 사전순 첫 쌍 하나만 담으세요. **empty_metrics** 에 both_empty(둘 다 빈 집합), no_prediction(예측이 빈 집합, 골드는 one_pair), no_gold(예측은 one_pair, 골드가 빈 집합)의 평가 결과를 담으세요.
- **metrics**, **overmerge_metrics**, **held_metrics** 를 출력해 FP(오병합 쌍 수)와 FN의 차이를 비교하세요.

**확인 기준**: 실제 그룹은 골드 37쌍에서 TP 37, FP 0, FN 0입니다. 가상 과병합은 FP가 늘고 정밀도가 낮아집니다. 충돌 그룹을 보류하면 FP는 0이지만 FN이 생깁니다. 골드의 ID 문자열과의 완전일치는 평가하지 않습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 서로 다른 처리 결과를 비교할 때도 골드와 전체 평가 범위는 고정합니다.

세부구현:
1. 그룹마다 모든 출현 쌍을 만듭니다.
2. 교집합과 차집합으로 TP, FP, FN을 셉니다.
3. 같은 함수로 실제 결과, 과병합과 보류 결과를 비교합니다.
4. 빈 집합은 정해진 분모 규칙을 따르는지 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert evaluation_ids == {row["mention_id"] for row in mentions}, "보류해도 평가 범위는 전체 출현입니다."
expected_gold = set()
for group in gold_data["groups"]:
    for left_id, right_id in combinations(sorted(set(group["mention_ids"]) & evaluation_ids), 2):
        expected_gold.add(pair_key(left_id, right_id))
assert gold_pairs == expected_gold, "골드는 과제 출현 범위로만 제한하세요."
for groups, pairs, result in [(identity_groups, predicted_pairs, metrics),
                              (proposed_groups, overmerge_pairs, overmerge_metrics),
                              (accepted_probe, held_pairs, held_metrics)]:
    expected_pairs = set()
    for group in groups:
        for left_id, right_id in combinations(group, 2):
            expected_pairs.add(pair_key(left_id, right_id))
    assert pairs == expected_pairs, "각 결과 그룹의 모든 쌍을 구하세요."
    tp, fp, fn = len(pairs & gold_pairs), len(pairs - gold_pairs), len(gold_pairs - pairs)
    expected = {"tp": tp, "fp": fp, "fn": fn,
                "precision": tp / (tp + fp) if tp + fp else 0.0,
                "recall": tp / (tp + fn) if tp + fn else 0.0,
                "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0}
    assert result == expected, "같은 골드에서 TP, FP, FN과 세 비율을 계산하세요."
    before = (set(pairs), set(gold_pairs))
    assert evaluate_pairs(pairs, gold_pairs) == expected, "모든 경우에 같은 평가 함수를 적용하세요."
    assert (pairs, gold_pairs) == before, "평가 입력을 수정하지 마세요."
assert metrics == {"tp": 37, "fp": 0, "fn": 0, "precision": 1.0, "recall": 1.0, "f1": 1.0}, "실제 판정 그룹의 결과를 확인하세요."
assert overmerge_metrics["fp"] > 0 and overmerge_metrics["precision"] < metrics["precision"], "다른 개체를 합치면 FP가 생깁니다."
assert held_metrics["fp"] == 0 and held_metrics["fn"] > 0, "보류한 같은 개체 쌍을 골드에서 빼면 안 됩니다."
assert one_pair == {min(gold_pairs)}, "사전순 첫 골드 쌍 하나를 담으세요."
expected_empty = {
    "both_empty": {"tp": 0, "fp": 0, "fn": 0, "precision": 0.0, "recall": 0.0, "f1": 0.0},
    "no_prediction": {"tp": 0, "fp": 0, "fn": 1, "precision": 0.0, "recall": 0.0, "f1": 0.0},
    "no_gold": {"tp": 0, "fp": 1, "fn": 0, "precision": 0.0, "recall": 0.0, "f1": 0.0},
}
assert empty_metrics == expected_empty, "빈 집합도 TP, FP, FN을 세고 분모가 0인 점수는 0.0으로 기록하세요."
assert evaluate_pairs(set(), set()) == expected_empty["both_empty"], "빈 집합도 같은 함수로 처리하세요."
assert evaluate_pairs(set(), one_pair) == expected_empty["no_prediction"], "예측이 없으면 정답 쌍이 FN입니다."
assert evaluate_pairs(one_pair, set()) == expected_empty["no_gold"], "골드가 없으면 추출한 쌍이 FP입니다."
print("✅ 통과!")


## Neo4j 연결을 준비합니다

이제 실습용 Neo4j와 APOC가 필요합니다. 아래 연결 셀을 실행한 뒤 9~10번을 진행하세요.  


In [ ]:
# [제공코드] 노드 초기화와 적재에 사용할 실습 전용 Neo4j에 연결합니다.
import os
from urllib.parse import urlsplit
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase

# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()

def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 목록으로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]

# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print("Neo4j 연결 완료. 호스트:", connection_address.hostname, "/ 포트:", connection_address.port)


In [ ]:
# [제공코드] 개체 목록을 원래 타입과 표준 ID로 저장하는 함수를 준비합니다.
def put_standard_nodes(nodes):
    """원래 타입을 라벨로 써서 표준 ID별 노드를 저장합니다.

    Args:
        nodes (list[dict]): entity_type, standard_id와 노드 속성이 있는 기록 목록.

    Returns:
        list[dict]: [{'written': 처리한 노드 수}]. 기존 노드 갱신도 포함합니다.
    """
    return run_cypher("""
    // 목록의 표준 개체를 하나씩 적재합니다.
    UNWIND $rows AS row
    // entity_type이 Book이면 Book 라벨을 사용합니다. 같은 타입과 ID의 노드는 재사용합니다.
    MERGE (n:$(row.entity_type) {standard_id: row.standard_id})
    // 표준 이름과 원래 출현 기록을 함께 보존합니다.
    SET n += row
    // 처리한 표준 개체 수를 반환합니다.
    RETURN count(n) AS written
    """, rows=nodes)


# 예시 입력: '파이썬 입문'과 '파이썬 입문서'를 같은 초판으로 모은 노드입니다.
node_example_nodes = [{"standard_id": "entity:d01:object", "canonical_name": "파이썬 입문 초판",
                  "entity_type": "Book", "aliases": ["파이썬 입문", "파이썬 입문서"]}]
# 예시 호출: 두 번 저장해도 같은 초판 노드를 재사용합니다.
# print(put_standard_nodes(node_example_nodes))
# print(put_standard_nodes(node_example_nodes))


# 이미 저장된 주어와 목적어 노드를 찾아 원래 관계를 연결하는 함수입니다.
# 등장한 자리마다 만든 노드와 같은 개체를 하나로 모은 노드에 모두 사용할 수 있습니다.
# 관계 타입에는 원래 relation을 쓰고, triple_id로 서로 다른 추출 행을 구분합니다.
def put_relations(rows, node_key):
    """주어 노드에서 목적어 노드로 관계를 저장하고 같은 트리플은 중복 생성하지 않습니다.

    Args:
        rows (list[dict]): 관계와 양 끝 타입이 있는 트리플. 표준 ID 적재에는 양 끝 ID도 필요합니다.
        node_key (str): 노드를 찾을 ID 속성. occurrence_id 또는 standard_id.

    Returns:
        list[dict]: [{'written': 처리한 관계 수}]. 같은 양 끝·타입·triple_id의
            기존 관계는 속성을 갱신하며, 양 끝 노드가 없는 행은 제외합니다.
    """
    # ID 속성 이름만 쿼리에 직접 넣습니다. 라벨은 각 행의 원래 타입을 사용합니다.
    if node_key not in {"occurrence_id", "standard_id"}:
        raise ValueError("ID 속성은 occurrence_id 또는 standard_id를 사용하세요.")

    records = []
    for row in rows:
        # 이미 만든 노드의 저장 방식에 맞춰 찾을 ID를 정합니다. 새 ID를 부여하지 않습니다.
        if node_key == "occurrence_id":
            # 등장한 자리로 찾기: d01은 d01:subject와 d01:object 노드를 연결합니다.
            # 같은 책도 d01:object와 d02:object라는 별도 노드로 저장된 상태입니다.
            subject_key = row["triple_id"] + ":subject"
            object_key = row["triple_id"] + ":object"
        else:
            # 확정한 개체 ID로 찾기: d01의 entity:d01:subject과 entity:d01:object 노드를 연결합니다.
            # d02의 책도 entity:d01:object이면 두 대출 관계가 같은 책 노드에 연결됩니다.
            subject_key = row["subject_id"]
            object_key = row["object_id"]
        records.append({"subject_key": subject_key, "object_key": object_key,
                        "properties": dict(row)})

    # 식별 속성에는 triple_id를 넣습니다. 같은 관계라도 출처 행이 다르면 보존합니다.
    query = f"""
    // 추출 행마다 원래 주어와 목적어의 식별자를 하나씩 처리합니다.
    UNWIND $rows AS item
    // 원래 주어와 목적어 타입을 라벨로 쓰고, 해당 ID의 기존 노드를 찾습니다.
    MATCH (s:$(item.properties.subject_type) {{{node_key}: item.subject_key}})
    MATCH (o:$(item.properties.object_type) {{{node_key}: item.object_key}})
    // $(...)는 각 행의 relation 값을 관계 타입으로 사용합니다.
    // 양 끝, 관계 타입과 triple_id가 같으면 기존 관계를 찾고, 없으면 만듭니다.
    MERGE (s)-[rel:$(item.properties.relation) {{triple_id: item.properties.triple_id}}]->(o)
    // 처음 저장하거나 다시 실행할 때 모두 원래 필드와 근거를 관계 속성에 기록합니다.
    SET rel += item.properties
    // 실제로 연결한 추출 행 수를 파이썬에서 확인할 수 있게 반환합니다.
    RETURN count(rel) AS written
    """
    return run_cypher(query, rows=records)


# 예시 입력: d01의 민수와 파이썬 입문 초판을 먼저 노드로 준비합니다.
relation_example_nodes = [{"standard_id": "entity:d01:subject", "canonical_name": "민수", "entity_type": "Person"},
                 {"standard_id": "entity:d01:object", "canonical_name": "파이썬 입문 초판", "entity_type": "Book"}]
relation_example_rows = [{"triple_id": "d01", "subject_id": "entity:d01:subject", "relation": "빌림",
                 "subject_type": "Person", "object_id": "entity:d01:object", "object_type": "Book",
                 "evidence": "민수는 김하나의 파이썬 입문 초판을 빌렸습니다."}]
# 예시 호출: 앞에서 준비한 put_standard_nodes로 양 끝 노드를 먼저 저장합니다.
# put_standard_nodes(relation_example_nodes)
# print(put_relations(relation_example_rows, "standard_id"))


## 9. 표준 노드와 원래 관계를 재실행해도 중복 없이 적재합니다

**배경**: 같은 추출 결과를 다시 적재해도 표준 노드와 원래 관계가 늘어나지 않아야 합니다.  

노드는 원래 타입인 `Document`, `ApiElement`, `Change`, `Issue`를 라벨로 사용합니다.  
제공 쿼리의 `$($row.entity_type)`은 노드 타입을, `$($row.relation)`은 관계 타입을 지정합니다.  
`graph_ids`는 준비 셀에서 만든 현재 자료의 표준 ID 목록입니다. 조회와 초기화는 이 범위에서만 수행합니다.  

**요구사항**  
- **load_graph()** 함수를 작성하세요. node_rows의 노드를 먼저 모두 적재하고, relationship_rows의 관계를 모두 적재합니다. 반환 값은 없습니다.
- **load_graph**의 노드 반복문에서는 각 노드의 mentions를 json.dumps로 직렬화하고, ensure_ascii=False를 지정하세요. Neo4j 속성에는 딕셔너리 목록 같은 중첩 구조를 그대로 넣을 수 없어 문자열로 바꿔 저장합니다. run_cypher에 node_upsert_query와 `row=해당 노드`, `mentions_json=직렬화한 문자열`을 전달하세요.
- **load_graph**의 관계 반복문에서는 해당 행의 relation이 allowed_relations에 포함되어 있는지 확인하세요. 포함되지 않으면 ValueError를 발생시키고, 포함되면 run_cypher에 relationship_upsert_query와 `row=해당 관계 행`을 전달하세요.
- **load_graph**는 allowed_relations를 호출할 때마다 다시 읽어야 합니다. 매개변수 기본값으로 묶어 두면 정의 시점의 값이 고정되어, 허용 목록이 바뀐 뒤 호출해도 예전 목록으로 검사합니다.
- **load_graph**를 정의한 뒤 아래 제공 실행 셀을 실행하세요. 직전 준비 단계에서 연결한 실습용 Neo4j에 같은 자료를 두 번 적재하고, 노드·관계 수와 원문 기록이 같은지 확인합니다.

**확인 기준**: 표준 노드 14개와 원래 관계 14개입니다. 두 번째 적재 뒤에도 같은 수와 같은 관계·근거가 남아야 합니다. 실제 자료의 14개 트리플을 모두 적재하며, 가상 오류 입력은 포함하지 않습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 교안에서 사용한 MERGE와 SET 쿼리는 제공되어 있습니다. 준비한 노드와 관계 자료를 빠짐없이 전달합니다.

세부구현:
1. node_rows를 순회하며 중첩된 출현 기록 목록을 JSON 문자열로 바꿉니다.
2. 원래 노드 행과 직렬화한 문자열을 노드 쿼리에 전달합니다.
3. relationship_rows를 순회하며 관계 타입을 확인하고 관계 쿼리에 전달합니다.
```

</details>


In [ ]:
# [제공코드] graph_ids는 현재 과제에서 적재할 표준 ID 목록입니다. 초기화와 조회에 사용합니다.
graph_ids = [row["standard_id"] for row in node_rows]

# 노드 라벨은 원래 entity_type, 관계 타입은 원래 relation을 사용합니다.
allowed_relations = {row["relation"] for row in original_triples}

# 노드는 표준 ID로 찾으므로 별칭이 늘어도 같은 노드를 사용합니다.
node_upsert_query = """
// 같은 표준 ID가 있으면 노드를 추가하지 않고 재사용합니다.
MERGE (n:$($row.entity_type) {standard_id: $row.standard_id})
// 표준 이름과 함께 원래 별칭, 출현 ID, 직렬화한 근거 기록도 저장합니다.
SET n.canonical_name = $row.canonical_name,
    n.entity_type = $row.entity_type,
    n.aliases = $row.aliases,
    n.mention_ids = $row.mention_ids,
    n.mentions_json = $mentions_json
// 어떤 표준 개체를 적재했는지 반환합니다.
RETURN n.standard_id AS standard_id
"""

# 원래 relation을 관계 타입으로 사용합니다. 같은 추출 행의 재실행은 triple_id로 찾습니다.
# $row에는 원래 표기, 원문과 저장 위치도 있으므로 생성과 재실행 때 모두 보존됩니다.
relationship_upsert_query = """
// 앞에서 적재한 두 표준 노드를 원래 주어와 목적어 방향으로 연결합니다.
MATCH (s:$($row.subject_type) {standard_id: $row.subject_id})
MATCH (o:$($row.object_type) {standard_id: $row.object_id})
// 같은 추출 행만 재사용하며, 다른 triple_id의 근거는 별도 관계로 남깁니다.
// $($row.relation)은 행에 기록된 관계 타입을 사용합니다(Neo4j 5.26 이상).
MERGE (s)-[rel:$($row.relation) {triple_id: $row.triple_id}]->(o)
// 생성과 재실행 모두 원래 필드와 근거를 관계 속성에 저장합니다.
SET rel += $row
// 실제로 적재한 추출 행을 확인할 수 있게 ID를 반환합니다.
RETURN rel.triple_id AS triple_id
"""
run_cypher("""
// 두 번 적재하기 전에 현재 과제의 표준 ID에 해당하는 노드만 초기화합니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids DETACH DELETE n
""", standard_ids=graph_ids)
print("원래 타입과 표준 ID로 적재할 개체 수:", len(graph_ids))


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
# DB에서 다시 읽은 실제 노드와 관계를 검사합니다. 파이썬 입력 자료만 검사하지 않습니다.
def read_graph_snapshot():
    """현재 자료의 노드와 관계를 원래 타입과 출처 속성까지 조회합니다."""
    nodes = run_cypher("""
    // 적재 입력과 비교할 실제 노드 속성을 읽습니다.
    MATCH (n)
    WHERE n.standard_id IN $standard_ids
    RETURN n.standard_id AS standard_id, labels(n) AS labels, n.canonical_name AS canonical_name,
           n.entity_type AS entity_type, n.aliases AS aliases,
           n.mention_ids AS mention_ids, n.mentions_json AS mentions_json
    // 조회 순서가 아니라 내용 차이로 비교하도록 ID 순서를 고정합니다.
    ORDER BY standard_id
    """, standard_ids=graph_ids)
    for node in nodes:
        # 실제 라벨도 원래 타입과 같아야 합니다. 비교용 결과에는 원래 속성만 남깁니다.
        assert node.pop("labels") == [node["entity_type"]], "노드 라벨을 원래 entity_type으로 저장하세요."
        # 직렬화 방법의 공백 차이가 아닌 원래 출현 기록의 내용으로 비교합니다.
        node["mentions"] = json.loads(node.pop("mentions_json"))
    relationships = run_cypher("""
    // 관계 속성에 적힌 ID만 믿지 않고 실제 양 끝 노드의 ID를 읽습니다.
    MATCH (s)-[r]->(o)
    WHERE s.standard_id IN $standard_ids AND o.standard_id IN $standard_ids
    // 원래 방향, 타입, 모든 근거 속성을 적재 입력과 대조합니다.
    RETURN s.standard_id AS source_id, type(r) AS relation,
           o.standard_id AS target_id, properties(r) AS stored
    ORDER BY r.triple_id
    """, standard_ids=graph_ids)
    return {"nodes": nodes, "relationships": relationships}

expected_relationships = []
for row in sorted(relationship_rows, key=lambda row: row["triple_id"]):
    stored = dict(row)
    expected_relationships.append({"source_id": row["subject_id"], "relation": row["relation"],
                                   "target_id": row["object_id"], "stored": stored})
expected_snapshot = {"nodes": node_rows, "relationships": expected_relationships}

load_graph()
first_snapshot = read_graph_snapshot()
assert first_snapshot == expected_snapshot, "실제 DB에서 표준 ID, 원래 관계 타입과 출처·근거가 모두 보존되었는지 확인하세요."
load_graph()
second_snapshot = read_graph_snapshot()
assert second_snapshot == expected_snapshot, "재실행 후에도 노드와 관계가 중복되거나 원래 속성이 달라지면 안 됩니다."
assert second_snapshot == first_snapshot, "두 적재 결과가 같아야 합니다."

# 허용 관계가 없는 조건에서도 적재한다면 관계 타입 검사가 빠진 것입니다.
allowed_before_check = allowed_relations
allowed_relations = set()
rejected_relation = False
try:
    load_graph()
except ValueError:
    rejected_relation = True
finally:
    allowed_relations = allowed_before_check
assert rejected_relation, "허용하지 않은 관계는 ValueError로 거부하세요. allowed_relations를 매개변수 기본값으로 묶지 말고 호출할 때 읽으세요."
print("✅ 통과! 두 번 적재 후 표준 노드:", len(second_snapshot["nodes"]),
      "/ 원래 관계:", len(second_snapshot["relationships"]))


## 기존 출현 노드를 통합하는 실습 준비

앞 문항은 처음부터 표준 ID로 노드를 생성했습니다. 이번에는 이미 출현별 노드가 저장된 경우를 연습합니다.  
아래 제공 셀은 현재 자료의 표준 ID에 해당하는 노드와 그 노드에 연결된 모든 관계를 비우고,  
적재 대상 28개 출현을 각각 노드로 만든 뒤 원래 14개 관계를 연결합니다.  
검사용 보류 결과는 이 적재에 섞지 않습니다. 반복하려면 이 준비 셀부터 다시 실행하세요.  


In [ ]:
# [제공코드] 원래 타입을 라벨로 쓰고, 같은 개체도 출현 ID마다 별도 노드로 준비합니다.
node_by_id = {row["standard_id"]: row for row in node_rows}
occurrence_nodes = []
for row in used_mentions:
    standard_id = linked[row["mention_id"]]
    occurrence_nodes.append({
        "occurrence_id": row["mention_id"], "standard_id": standard_id,
        "canonical_name": node_by_id[standard_id]["canonical_name"],
        "entity_type": row["entity_type"], "aliases": [row["name"]],
        "mention_ids": [row["mention_id"]], "source_doc_ids": [row["source_doc_id"]],
    })

# 설정은 다음 답안에서 작성합니다. graph_ids 범위에서 같은 확정 ID끼리 모읍니다.
merge_nodes_query = """
// 현재 자료의 표준 ID에 해당하는 노드만 찾습니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids
// 통합 후 남길 첫 노드를 일정하게 선택합니다.
WITH n ORDER BY n.occurrence_id
// 이름 유사도가 아니라 확정한 표준 ID가 같은 노드만 모읍니다.
WITH n.standard_id AS standard_id, collect(n) AS nodes
WHERE size(nodes) > 1
// 학생이 정한 정책으로 속성을 보존하며 관계를 통합 노드로 옮깁니다.
CALL apoc.refactor.mergeNodes(nodes, $config) YIELD node
// 실제로 통합한 그룹만 반환하므로 두 번째 실행 결과는 빈 목록입니다.
RETURN standard_id, node.mention_ids AS mention_ids
"""
run_cypher("""
// 앞 문항의 현재 자료 노드를 비우고, 출현별 저장 상태를 만듭니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids DETACH DELETE n
""", standard_ids=graph_ids)
run_cypher("""
// 표준 ID가 같아도 우선은 출현마다 노드를 따로 만듭니다.
UNWIND $rows AS row
MERGE (n:$(row.entity_type) {occurrence_id: row.occurrence_id})
// 통합 뒤 원래 별칭과 출처를 확인할 수 있게 모든 준비 속성을 남깁니다.
SET n += row
""", rows=occurrence_nodes)
put_relations(normalized_triples, "occurrence_id")
before_nodes = run_cypher("""
// 각 출현이 통합 전에는 서로 다른 노드로 존재하는지 셉니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids RETURN count(n) AS count
""", standard_ids=graph_ids)
assert before_nodes[0]["count"] == len(used_mentions), "준비 단계는 출현마다 노드를 하나씩 만듭니다."
print("통합 전 노드:", before_nodes[0]["count"])


## 10. APOC로 기존 노드를 통합하고 관계 이동을 확인합니다

**배경**: 출현별로 이미 생성한 중복 노드를 합쳐도 각 추출 관계와 근거를 보존해야 합니다.  

**요구사항**  
- **merge_config** 는 properties, mergeRels, singleElementAsArray 세 키만 가진 딕셔너리입니다. properties에는 aliases·mention_ids·source_doc_ids의 정책을 combine, 나머지 속성에 적용할 `.*` 정책을 discard로 지정합니다. properties에는 위 네 키만 사용합니다. mergeRels는 False, singleElementAsArray는 True로 지정하세요.
- **merge_duplicates()** 함수를 작성하세요. run_cypher에 제공된 merge_nodes_query, `config=merge_config`, `standard_ids=graph_ids`를 전달하고 조회 결과를 그대로 반환하세요. 함수 정의만 하고 실행은 아래 제공 자가채점 셀에서 수행합니다.
- **merge_config** 와 **merge_duplicates** 의 결과로 같은 표준 ID의 노드는 하나가 되고 원래 타입 라벨과 14개 관계의 방향·타입·모든 속성이 유지되어야 합니다. 아래 실제 DB 검사에서 출현 소속까지 확인하세요.

**확인 기준**: 설정은 DB 없이 검사합니다. 실제 DB에서는 28개 출현 노드가 14개 표준 노드로 통합되고, 관계 14개가 각각 남습니다. 두 번 통합해도 결과가 같습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 속성 병합 정책과 관계 보존 옵션을 사용합니다.

세부구현:
1. 여러 출현의 값을 보존할 목록 속성에 결합 정책을 지정합니다.
2. 관계를 합치지 않고 한 값도 목록으로 남기는 옵션을 지정합니다.
3. 제공 쿼리에 통합 설정을 전달합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert set(merge_config) == {"properties", "mergeRels", "singleElementAsArray"}, "설정의 세 키를 확인하세요."
assert merge_config["properties"] == {"aliases": "combine", "mention_ids": "combine", "source_doc_ids": "combine", ".*": "discard"}, "목록 속성은 결합하고 나머지는 첫 값을 유지하세요."
assert merge_config["mergeRels"] is False, "원래 추출 행별 관계를 보존하세요."
assert merge_config["singleElementAsArray"] is True, "값 하나도 목록 형식을 유지하세요."
assert callable(merge_duplicates), "merge_duplicates 함수를 정의하세요."
print("✅ 설정 통과! 실제 노드 통합은 다음 DB 검사에서 확인합니다.")


In [ ]:
# [자가채점]
# 실제 관계의 양 끝을 DB 노드 식별자로 비교하여 통합하지 않은 오답을 구분합니다.
expected_by_id = {row["triple_id"]: row for row in normalized_triples}
expected_members = {row["standard_id"]: set(row["mention_ids"]) for row in node_rows}
# 출현 ID로 바로 조회하도록 준비해 노드마다 원본 목록 전체를 검색하지 않습니다.
used_mention_by_id = {row["mention_id"]: row for row in used_mentions}
for repeat in range(2):
    # 첫 통합은 중복 그룹을, 두 번째 통합은 처리할 그룹이 없는 빈 목록을 반환합니다.
    merge_result = merge_duplicates()
    assert isinstance(merge_result, list), "쿼리의 조회 결과 목록을 반환하세요."
    expected_merged = {}
    if repeat == 0:
        expected_merged = {key: members for key, members in expected_members.items() if len(members) > 1}
    assert all(set(row) == {"standard_id", "mention_ids"} for row in merge_result), "조회 결과의 두 필드를 그대로 반환하세요."
    returned_members = {row["standard_id"]: set(row["mention_ids"]) for row in merge_result}
    assert len(merge_result) == len(expected_merged) and returned_members == expected_merged, "실제로 통합한 그룹의 조회 결과를 반환하세요."
    actual_nodes = run_cypher("""
    // 표준 ID만 같고 실제로는 별도 노드인 경우를 구분하려고 elementId도 읽습니다.
    MATCH (n)
    WHERE n.standard_id IN $standard_ids
    RETURN elementId(n) AS node_id, n.standard_id AS standard_id, labels(n) AS labels,
           n.mention_ids AS mention_ids, n.aliases AS aliases,
           n.source_doc_ids AS source_doc_ids
    """, standard_ids=graph_ids)
    actual_edges = run_cypher("""
    // 관계가 실제로 옮겨진 시작점과 도착점을 따라 출현 소속을 다시 계산합니다.
    MATCH (s)-[r]->(o)
    WHERE s.standard_id IN $standard_ids AND o.standard_id IN $standard_ids
    // 원래 관계 타입과 모든 근거 속성이 보존됐는지도 함께 검사합니다.
    RETURN elementId(s) AS source_node, elementId(o) AS target_node,
           s.standard_id AS subject_id, o.standard_id AS object_id,
           type(r) AS relation, properties(r) AS stored
    """, standard_ids=graph_ids)
    assert len(actual_nodes) == len(expected_members), "같은 표준 ID의 실제 노드를 하나로 합치세요."
    assert {row["standard_id"] for row in actual_nodes} == set(expected_members), "표준 ID를 보존하세요."
    actual_members = {}
    seen_triples = []
    for edge in actual_edges:
        triple_id = edge["stored"]["triple_id"]
        assert triple_id in expected_by_id, "원래 자료에 없는 관계를 만들지 마세요."
        original = expected_by_id[triple_id]
        seen_triples.append(triple_id)
        expected_properties = dict(original)
        assert edge["stored"] == expected_properties, "각 추출 관계의 모든 속성과 근거를 보존하세요."
        assert edge["relation"] == original["relation"], "원래 관계 타입을 유지하세요."
        assert edge["subject_id"] == original["subject_id"] and edge["object_id"] == original["object_id"], "관계의 방향과 표준 ID를 확인하세요."
        actual_members.setdefault(edge["source_node"], set()).add(triple_id + ":subject")
        actual_members.setdefault(edge["target_node"], set()).add(triple_id + ":object")
    assert len(seen_triples) == len(set(seen_triples)) == len(expected_by_id), "관계가 누락되거나 합쳐지거나 중복되면 안 됩니다."
    assert set(seen_triples) == set(expected_by_id), "원래 모든 추출 행을 보존하세요."
    for node in actual_nodes:
        expected_type = node_by_id[node["standard_id"]]["entity_type"]
        assert node["labels"] == [expected_type], "통합 후에도 원래 타입만 라벨로 남아야 합니다."
        members = expected_members[node["standard_id"]]
        assert actual_members.get(node["node_id"]) == members, "실제 관계의 양 끝이 올바른 통합 노드로 이동해야 합니다."
        assert len(node["mention_ids"]) == len(set(node["mention_ids"])) and set(node["mention_ids"]) == members, "출현 ID를 누락·중복 없이 보존하세요."
        originals = [used_mention_by_id[mention_id] for mention_id in members]
        assert isinstance(node["aliases"], list) and set(node["aliases"]) == {row["name"] for row in originals}, "별칭은 원래 표기의 목록으로 보존하세요."
        assert isinstance(node["source_doc_ids"], list) and set(node["source_doc_ids"]) == {row["source_doc_id"] for row in originals}, "출처 문서는 목록으로 보존하세요."
print("✅ 통과! 두 번 통합 후 실제 노드와 관계, 출현 소속, 원문 근거가 같습니다.")
